In [ ]:
%load_ext autoreload
%autoreload 2
%autosave 90

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Autosaving every 30 seconds


In [ ]:
import hydra
import torch
from omegaconf import DictConfig

from genpp import BASE_DIR
from genpp.configs import register_resolvers
from genpp.eval.FID import fid
from genpp.eval.FID.autoencoder import AutoEncoder

try:
    register_resolvers()
except Exception:
    pass

# Load the Autoencoder

In [81]:
# Model ID is generated by WandB
RUN_PATH = "feik/genpp/nyuw6ut1"
MODEL_ID = RUN_PATH.split("/")[-1]
OUTPUT_DIR = BASE_DIR.parent.parent / "outputs"

MODEL_DIR = list(OUTPUT_DIR.rglob(f"*{MODEL_ID}*"))[0].parent.parent.parent
MODEL_CHECKPOINT = list(MODEL_DIR.rglob("*.ckpt"))[0]

In [82]:
ae = AutoEncoder.load_from_checkpoint(MODEL_CHECKPOINT)

Ignored args: (), kwargs: {'use_rescaler': False, 'rescaler': None}
Loading channel_means from checkpoint
Loading channel_stds from checkpoint


# Compute the mean and std for the real Images

In [55]:
# Load the dataset
# Load model and dataloader to get predictions
with hydra.initialize_config_dir(config_dir=str(MODEL_DIR / ".hydra"), version_base=None):
    cfg: DictConfig = hydra.compose(
        config_name="config",
    )
# Compatibility fix for old configs
if cfg.data.module._target_ == "genpp.data.zarr.ZarrDataModule":
    cfg.data.module._target_ = "genpp.data.zarr_dataset.ZarrDataModule"

datamodule = hydra.utils.instantiate(cfg.data.module)
datamodule.prepare_data()
datamodule.setup(stage="validate")
datamodule.setup(stage="test")

In [ ]:
trainer = hydra.utils.instantiate(cfg.trainer, logger=False)

predictions_val = trainer.predict(ae, datamodule.val_dataloader())
predictions_cat_val = torch.cat(predictions_val, dim=0)

predictions_test = trainer.predict(ae, datamodule.test_dataloader())
predictions_cat_test = torch.cat(predictions_test, dim=0)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting DataLoader 0: 100%|██████████| 1426/1426 [00:05<00:00, 240.38it/s]


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting DataLoader 0: 100%|██████████| 1426/1426 [00:05<00:00, 250.19it/s]


In [57]:
means_real_val = predictions_cat_val.mean(dim=0)
covs_real_val = torch.cov(predictions_cat_val.T)

means_real_test = predictions_cat_test.mean(dim=0)
covs_real_test = torch.cov(predictions_cat_test.T)

### Sanity check, compute the FID distance for the images from the test and the val set
This should be a very low (lower is better)

In [68]:
res = fid(features1=predictions_cat_test, features2=predictions_cat_val)
res

0.02744138240814209

## Load the postprocessing Model and compute the statistics
Lets start with the chen model for now.

Chen model: `feik/genpp/qbuvhf5p`

In [ ]:
import importlib

import lightning as L
from einops import rearrange
from torch.utils.data import DataLoader, TensorDataset

from genpp.configs import add_y_kwargs, del_key
from genpp.models.cgm.chen import CNNChenModel

In [ ]:
RUN_PATH = "feik/genpp/blkpcik8"
MODEL_ID = RUN_PATH.split("/")[-1]
OUTPUT_DIR = BASE_DIR.parent.parent / "outputs"

MODEL_DIR = list(OUTPUT_DIR.rglob(f"*{MODEL_ID}*"))[0].parent.parent.parent
MODEL_CHECKPOINT = list(MODEL_DIR.rglob("*.ckpt"))[0]

SCORE_FILE = MODEL_DIR / "scores.csv"

with hydra.initialize_config_dir(config_dir=str(MODEL_DIR / ".hydra"), version_base=None):
    cfg: DictConfig = hydra.compose(
        config_name="config",
    )
# Do not shuffle any dataloader
cfg.data.module.dataloader_config.train.shuffle = False
cfg.data.module.dataloader_config.val.shuffle = False
cfg.data.module.dataloader_config.val.batch_size = 16  # For faster predictions
cfg.data.module.dataloader_config.test.shuffle = False
add_y_kwargs(
    cfg, y_kwargs={"batch_dims": {}, "input_dims": {"feature": 2, "longitude": 37, "latitude": 31}}
)
del_key(cfg.data.module.dataset_config.train.x_kwargs.batch_dims, "time")
del_key(cfg.data.module.dataset_config.train.x_kwargs.batch_dims, "prediction_timedelta")

datamodule = hydra.utils.instantiate(cfg.data.module)
datamodule.prepare_data()
datamodule.setup(stage="validate")
datamodule.setup(stage="test")

class_path = cfg.model._target_

module_path, class_name = class_path.rsplit(".", 1)
module = importlib.import_module(module_path)
ModelClass = getattr(module, class_name)

if ModelClass is CNNChenModel:
    model = ModelClass.load_from_checkpoint(
        MODEL_CHECKPOINT,
        final_activation=hydra.utils.instantiate(cfg.model.final_activation),
        loss_fn=hydra.utils.instantiate(cfg.model.loss_fn),
    )
else:
    model = ModelClass.load_from_checkpoint(MODEL_CHECKPOINT)

trainer = L.Trainer(logger=False, accelerator="gpu", devices=[0])

Adding y_kwargs to dataset_config.train
Deleted key 'time' from config.
Deleted key 'prediction_timedelta' from config.
Configuration hash: 980620c515fdadefc42dff7cfb4e72f8827ffcc7ce573927f07acac73dffa36d
Cached tensor data found. Verifying configuration...
Using cached tensor data from /home/feik/GenPP/src/genpp/data/weatherbench2/cache/tensor_980620c515fdadefc42dff7cfb4e72f8827ffcc7ce573927f07acac73dffa36d.pt.


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


Loading scale_variance_td from checkpoint


In [78]:
pred_list = trainer.predict(model, datamodule.val_dataloader(), return_predictions=True)
predictions = torch.cat(pred_list, dim=0)  # type: ignore

# Rescale the predictions (TODO this should happen in the predict step)
reverse_transform = datamodule.y_reverseModules[0]
mean = rearrange(reverse_transform.mean, "f -> 1 1 f 1 1")
scale = rearrange(reverse_transform.scale, "f -> 1 1 f 1 1")

predictions_rescaled = predictions * scale + mean

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting DataLoader 0: 100%|██████████| 227/227 [12:20<00:00,  0.31it/s]


In [85]:
predictions_rescaled_reshaped = rearrange(predictions_rescaled, "t n c h w -> (t n) c h w")

In [ ]:
# Feed the rescaled predictions to FID computation
predictions_ds = TensorDataset(predictions_rescaled_reshaped)
predictions_dl = DataLoader(predictions_ds, batch_size=128, shuffle=False)

predictions_model: list[torch.Tensor] = trainer.predict(ae, predictions_dl)  # type: ignore
predictions_cat = torch.cat(predictions_model, dim=0)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting DataLoader 0: 100%|██████████| 1415/1415 [00:04<00:00, 303.58it/s]


### Compute the fid

In [89]:
# TODO make sure the sets are somewhat similar as this will probably impact the FID
fid_score = fid(features1=predictions_cat, features2=predictions_cat_val)
fid_score

0.008709430694580078